# 1: Data Loading & Preparation
## RAG-Based Medical QA System — PubMedQA Corpus
---
**What this notebook does:**
- Installs required libraries
- Loads PubMedQA dataset from HuggingFace
- Explores and understands the data structure
- Cleans and extracts documents (abstracts)
- Saves cleaned corpus as JSON for next steps


## 1.1 Install Required Libraries

In [ ]:
!pip install datasets pandas tqdm -q

## 1.2 Import Libraries

In [ ]:
from datasets import load_dataset
import pandas as pd
import json
import re
from tqdm import tqdm

print('All libraries imported successfully!')

All libraries imported successfully!


## 1.3 Load PubMedQA Dataset

We load two subsets:
- `pqa_labeled` → 1,000 expert-labeled samples (we'll use these as our **test queries** for evaluation)
- `pqa_unlabeled` → 61,000 samples (we'll use abstracts from these as our **corpus/documents**)

In [ ]:
print('Loading pqa_labeled (for test queries)...')
labeled_ds = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', trust_remote_code=True)
print(f'pqa_labeled loaded: {labeled_ds}')

print('\nLoading pqa_unlabeled (for corpus)...')
unlabeled_ds = load_dataset('qiaojin/PubMedQA', 'pqa_unlabeled', trust_remote_code=True)
print(f'pqa_unlabeled loaded: {unlabeled_ds}')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading pqa_labeled (for test queries)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

...

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


pqa_labeled loaded: DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})

Loading pqa_unlabeled (for corpus)...


pqa_unlabeled/train-00000-of-00001.parqu(…):   0%|          | 0.00/66.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61249 [00:00<?, ? examples/s]

pqa_unlabeled loaded: DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer'],
        num_rows: 61249
    })
})


## 1.4 Explore Data Structure

Let's understand exactly what each record looks like before we process it.

In [ ]:
# Look at one labeled example in detail
sample = labeled_ds['train'][0]

print('=== KEYS IN EACH RECORD ===')
print(list(sample.keys()))

print('\n=== PUBMED ID ===')
print(sample['pubid'])

print('\n=== QUESTION ===')
print(sample['question'])

print('\n=== CONTEXT (Abstract sentences) ===')
for i, sentence in enumerate(sample['context']['contexts']):
    print(f'  [{i}] {sentence[:120]}...')

print('\n=== LONG ANSWER ===')
print(sample['long_answer'])

print('\n=== FINAL DECISION ===')
print(sample['final_decision'])

=== KEYS IN EACH RECORD ===
['pubid', 'question', 'context', 'long_answer', 'final_decision']

=== PUBMED ID ===
21645374

=== QUESTION ===
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

=== CONTEXT (Abstract sentences) ===
  [0] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascarien...
  [1] The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. mad...

=== LONG ANSWER ===
Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the first time, we have shown the feasibility for the use of

In [ ]:
# Check distribution of final decisions in labeled set
decisions = [item['final_decision'] for item in labeled_ds['train']]
from collections import Counter
print('=== Decision Distribution in Labeled Set ===')
print(Counter(decisions))

print(f'\nTotal labeled samples: {len(labeled_ds["train"])}')
print(f'Total unlabeled samples: {len(unlabeled_ds["train"])}')

=== Decision Distribution in Labeled Set ===
Counter({'yes': 552, 'no': 338, 'maybe': 110})

Total labeled samples: 1000
Total unlabeled samples: 61249


## 1.5 Build the Document Corpus

We extract abstracts from the unlabeled set.
Each record's `context['contexts']` is a list of sentences — we join them into one full abstract per document.

We'll use **5,000 documents** from the unlabeled set — more than enough for 500+ chunks.

In [ ]:
def clean_text(text):
    """Basic text cleaning."""
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Strip leading/trailing whitespace
    text = text.strip()
    return text

def extract_documents(dataset_split, max_docs=5000):
    """
    Extract and clean documents from PubMedQA dataset.
    Returns a list of dicts with: doc_id, pubid, question, abstract, answer
    """
    documents = []

    for i, item in enumerate(tqdm(dataset_split, desc='Extracting documents')):
        if i >= max_docs:
            break

        # Join all abstract sentences into one paragraph
        abstract_sentences = item['context']['contexts']
        full_abstract = ' '.join(abstract_sentences)
        full_abstract = clean_text(full_abstract)

        # Skip if abstract is too short (likely bad data)
        if len(full_abstract.split()) < 30:
            continue

        question = clean_text(item['question'])
        long_answer = clean_text(item['long_answer']) if item['long_answer'] else ''

        doc = {
            'doc_id': f'pubmed_{i:05d}',
            'pubid': str(item['pubid']),
            'question': question,
            'abstract': full_abstract,
            'long_answer': long_answer,
            'word_count': len(full_abstract.split())
        }
        documents.append(doc)

    return documents

print('Extracting documents from unlabeled set...')
corpus_docs = extract_documents(unlabeled_ds['train'], max_docs=5000)
print(f'\nTotal documents extracted: {len(corpus_docs)}')

Extracting documents from unlabeled set...


Extracting documents:   8%|▊         | 5000/61249 [00:03<00:42, 1315.90it/s]


Total documents extracted: 5000


## 1.6 Explore the Corpus Statistics

In [ ]:
# Word count statistics
word_counts = [doc['word_count'] for doc in corpus_docs]

df_stats = pd.DataFrame({'word_count': word_counts})

print('=== CORPUS STATISTICS ===')
print(f'Total Documents : {len(corpus_docs)}')
print(f'Min word count  : {df_stats["word_count"].min()}')
print(f'Max word count  : {df_stats["word_count"].max()}')
print(f'Mean word count : {df_stats["word_count"].mean():.1f}')
print(f'Median word count: {df_stats["word_count"].median():.1f}')

print('\n=== SAMPLE DOCUMENT ===')
sample_doc = corpus_docs[42]
print(f'Doc ID   : {sample_doc["doc_id"]}')
print(f'PubMed ID: {sample_doc["pubid"]}')
print(f'Question : {sample_doc["question"]}')
print(f'Abstract : {sample_doc["abstract"][:300]}...')
print(f'Words    : {sample_doc["word_count"]}')

=== CORPUS STATISTICS ===
Total Documents : 5000
Min word count  : 33
Max word count  : 606
Mean word count : 200.7
Median word count: 197.0

=== SAMPLE DOCUMENT ===
Doc ID   : pubmed_00042
PubMed ID: 14511046
Question : Open mini-access ureterolithotomy: the treatment of choice for the refractory ureteric stone?
Abstract : To report the experience in one centre of the efficacy and safety of open mini-access ureterolithotomy (MAU) and to discuss relevant current indications. MAU was undertaken in 112 patients (mean age 38 years, range 26-57) between 1991 and 2001; the details and outcomes are reviewed. The mean (range)...
Words    : 184


## 1.7 Prepare Test Queries (from Labeled Set)

We extract 20 queries from the labeled set for our **LLM-as-a-Judge evaluation pipeline** later.

In [ ]:
def extract_test_queries(labeled_split, n=20):
    """
    Extract test queries with ground truth answers for evaluation.
    """
    test_queries = []

    for i, item in enumerate(labeled_split):
        if i >= n:
            break

        query = {
            'query_id': f'query_{i:03d}',
            'question': clean_text(item['question']),
            'ground_truth_answer': clean_text(item['long_answer']),
            'final_decision': item['final_decision'],
            'reference_abstract': ' '.join(item['context']['contexts'])
        }
        test_queries.append(query)

    return test_queries

test_queries = extract_test_queries(labeled_ds['train'], n=20)
print(f'Test queries prepared: {len(test_queries)}')

print('\n=== SAMPLE TEST QUERY ===')
print(f'Question: {test_queries[0]["question"]}')
print(f'Answer  : {test_queries[0]["ground_truth_answer"][:200]}...')
print(f'Decision: {test_queries[0]["final_decision"]}')

Test queries prepared: 20

=== SAMPLE TEST QUERY ===
Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Answer  : Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of ...
Decision: yes


## 1.8 Save Everything to JSON

We save both files — the corpus and test queries — for use in all subsequent steps.

In [ ]:
# Save corpus
corpus_path = 'pubmed_corpus.json'
with open(corpus_path, 'w', encoding='utf-8') as f:
    json.dump(corpus_docs, f, indent=2, ensure_ascii=False)
print(f'Corpus saved → {corpus_path} ({len(corpus_docs)} documents)')

# Save test queries
queries_path = 'test_queries.json'
with open(queries_path, 'w', encoding='utf-8') as f:
    json.dump(test_queries, f, indent=2, ensure_ascii=False)
print(f'Test queries saved → {queries_path} ({len(test_queries)} queries)')

print('\nStep 1 Complete!')
print('Next step: Chunking strategies (Fixed vs Recursive)')

Corpus saved → pubmed_corpus.json (5000 documents)
Test queries saved → test_queries.json (20 queries)

Step 1 Complete!
Next step: Chunking strategies (Fixed vs Recursive)


## 1.9 Download Files to Your Local Machine

Run this cell to download both JSON files from Colab to your computer.
Keep them safe — you'll upload them again in future steps.

In [ ]:
from google.colab import files
files.download('pubmed_corpus.json')
files.download('test_queries.json')
print('Files downloaded!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files downloaded!
